# Demo — Encoding HIPAA & Medallion Policy in a Data Contract
In Veridian Health's Medallion Lakehouse, promoting a dataset from **Bronze** to **Silver** requires a machine-validated **Data Contract**. Here we inspect the starting contract for `silver_clinical_encounter` (Clinical Telehealth) and encode its HIPAA retention (`P6Y` per 45 CFR §164.530(j)), linkable PHI column (`mpi_id`), and Silver quality SLOs.

In [1]:
import json
from demo_toolkit import base_encounter_contract, validate_medallion_contract, build_medallion_lineage

encounter_contract = base_encounter_contract()
print(json.dumps(encounter_contract, indent=2))
valid, errors = validate_medallion_contract(encounter_contract)
print(f"\nValid? {valid} | Errors: {errors}")

{
  "dataset": "silver_clinical_encounter",
  "medallion_zone": "silver",
  "domain": "Clinical Telehealth",
  "steward": "clinical-ops@veridian.health",
  "version": "1.0.0",
  "columns": [
    {
      "name": "encounter_id",
      "type": "integer"
    },
    {
      "name": "mpi_id",
      "type": "integer"
    },
    {
      "name": "provider_npi",
      "type": "string"
    },
    {
      "name": "encounter_ts",
      "type": "timestamp"
    },
    {
      "name": "icd10_code",
      "type": "string"
    },
    {
      "name": "acuity_level",
      "type": "string"
    },
    {
      "name": "copay_usd",
      "type": "double"
    },
    {
      "name": "session_research_opt_in",
      "type": "integer"
    }
  ]
}

Valid? False | Errors: ["(root): 'governance' is a required property"]


## 1. What the validator checks (JSON Schema + semantic PHI/SLO rules)

Watch what happens if we supply an invalid retention string (`"6_years"` instead of ISO-8601 `"P6Y"`), a percentage threshold outside `(0, 1]` (`99.0`), or list `"mpi_id"` in `phi_columns` without marking `col["phi"] = True` on the column definition:

In [2]:
draft = base_encounter_contract()
draft["governance"] = {
    "phi_columns": ["mpi_id"],
    "retention_policy": "6_years",          # invalid: must match ^P\d+[YMD]$
    "regulatory_regime": ["HIPAA", "GDPR-Art9"],
    "quality_slos": [{"dimension": "completeness", "threshold": 99.0}],  # invalid: must be in (0, 1]
}
valid, errors = validate_medallion_contract(draft)
print("Valid?", valid)
for e in errors:
    print(" -", e)

Valid? False
 - governance/quality_slos/0/threshold: 99.0 is greater than the maximum of 1.0
 - governance/retention_policy: '6_years' does not match '^P\\d+[YMD]$'
 - columns: at least one PHI/linkable column must be marked with col['phi'] = True


## 2. Construct a compliant Silver contract & trace downstream propagation

Now mark `mpi_id` with `col["phi"] = True`, set HIPAA 6-year retention (`"P6Y"`), and specify normalized SLO thresholds in `(0, 1]`:

In [3]:
governed = base_encounter_contract()
governed["columns"][1]["phi"] = True  # mpi_id links the clinical encounter to silver_patient_master
governed["governance"] = {
    "phi_columns": ["mpi_id"],
    "retention_policy": "P6Y",
    "regulatory_regime": ["HIPAA-164.530", "GDPR-Art9"],
    "quality_slos": [
        {"dimension": "completeness", "column": "icd10_code", "threshold": 0.99},
        {"dimension": "accuracy", "column": "copay_usd", "threshold": 0.999},
        {"dimension": "consistency", "column": "session_research_opt_in", "threshold": 0.99},
    ],
}
valid, errors = validate_medallion_contract(governed)
print("Valid?", valid, "| Errors:", errors)
lineage = build_medallion_lineage()
print("Downstream Gold consumers of silver_clinical_encounter:", lineage.trace_downstream("silver_clinical_encounter"))

Valid? True | Errors: []
Downstream Gold consumers of silver_clinical_encounter: ['gold_encounter_turnaround', 'gold_deidentified_research_cohort', 'gold_cms_timely_care']


**Policy propagation across the Medallion zones:**
- `gold_deidentified_research_cohort` joins `silver_clinical_encounter` with `gold_patient_clinical_360` — it inherits the PHI and HIPAA research-authorization obligations (`mpi_id` must be de-identified and rows without `hipaa_research_auth = 1` excluded).
- `gold_encounter_turnaround` and `gold_cms_timely_care` aggregate encounter timestamps and lab wait times without `mpi_id` (no PHI inherited), but rely on `silver_clinical_encounter`'s `P6Y` retention window to keep multi-year CMS eCQM disclosures auditable.